In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 4.4 — Building End-to-End ML Solutions

**Objectives**

By the end of this section you will be able to:

- Chain preprocessing and modelling into a reusable sklearn `Pipeline`.
- Tune hyperparameters systematically using `GridSearchCV`.
- Save and reload a trained model using `joblib`.
- Identify key indicators for monitoring a deployed model in production.

> **Cooking analogy:** A model that lives only in a notebook is like a recipe
> that never leaves the test kitchen — impressive in private, useless in
> practice. **Deploying** a model means putting it on the restaurant menu:
> standardising the process so any cook can follow it (Pipeline), printing the
> recipe card for long-term use (saved model file), and checking each night that
> the dish still tastes the way it should (performance monitoring). Good accuracy
> in testing means nothing if the dish falls apart when it reaches the customer.

Delivering an ML model as a business solution requires more than good accuracy.
This section covers pipelines, model persistence, and performance monitoring.

### Sklearn Pipelines

A **Pipeline** chains preprocessing and modelling into a single reusable object
— like a **kitchen assembly line**: raw ingredients enter at one end and pass
through washing, chopping, cooking, and plating stations in a fixed sequence.
A finished dish emerges at the other end every time, with no steps skipped or
performed out of order.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
n = 800

# Re-create loan dataset
df_loan_final = pd.DataFrame({
    "Income"         : np.random.normal(55_000, 20_000, n).clip(20_000, 150_000).round(-2),
    "Loan_Amount"    : np.random.normal(25_000, 10_000, n).clip(5_000, 80_000).round(-2),
    "Credit_Score"   : np.random.randint(500, 850, n),
    "Age"            : np.random.randint(22, 65, n),
    "Employment_Yrs" : np.random.randint(0, 30, n),
})
df_loan_final["Approved"] = (
    (df_loan_final["Credit_Score"] > 650) &
    (df_loan_final["Income"] > 40_000)
).astype(int)

X = df_loan_final.drop(columns="Approved")
y = df_loan_final["Approved"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Build pipeline: scale → classify
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(n_estimators=100, random_state=42))
])

pipe.fit(X_tr, y_tr)
y_pred_pipe = pipe.predict(X_te)

from sklearn.metrics import accuracy_score
print(f"Pipeline Accuracy: {accuracy_score(y_te, y_pred_pipe):.2%}")

### Hyperparameter Tuning with GridSearchCV

> **Cooking analogy:** `GridSearchCV` is a **structured tasting panel**. You
> systematically try every combination of oven temperature and cooking time, ask
> a panel of judges (cross-validation folds) to score each result, and choose
> the combination that consistently scores highest. It replaces guesswork with
> methodical, repeatable experimentation.

In [ ]:
param_grid = {
    "clf__n_estimators" : [50, 100, 200],
    "clf__max_depth"    : [3, 5, None],
}

gs = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(X_tr, y_tr)

print("Best parameters:", gs.best_params_)
print(f"Best CV Accuracy: {gs.best_score_:.2%}")
print(f"Test Accuracy   : {gs.score(X_te, y_te):.2%}")

### Saving and Loading Models

> **Cooking analogy:** Saving a model with `joblib` is like **laminating the
> winning recipe card** and pinning it permanently to the kitchen wall. Any
> cook — today, next month, or two years from now — can pick it up and reproduce
> the dish exactly. Without saving, all that training effort vanishes the moment
> you close the notebook.

In [ ]:
import joblib

# Save the trained pipeline to disk
joblib.dump(gs.best_estimator_, "loan_approval_model.pkl")
print("Model saved to loan_approval_model.pkl")

# Load and use the model
loaded_model = joblib.load("loan_approval_model.pkl")

# Predict for a new applicant
new_applicant = pd.DataFrame([{
    "Income"         : 62_000,
    "Loan_Amount"    : 20_000,
    "Credit_Score"   : 710,
    "Age"            : 35,
    "Employment_Yrs" : 8
}])

prediction  = loaded_model.predict(new_applicant)[0]
probability = loaded_model.predict_proba(new_applicant)[0, 1]

outcome = "APPROVED" if prediction == 1 else "DENIED"
print(f"\nLoan Application Decision: {outcome}")
print(f"Approval Probability: {probability:.2%}")

### Model Monitoring Checklist

Once a model is deployed, track these indicators:

| Indicator | What to Monitor | Alert Threshold |
|---|---|---|
| **Accuracy drift** | Monthly accuracy vs baseline | Drop > 5 % |
| **Data drift** | Distribution shift in features | KS-test p < 0.05 |
| **Prediction drift** | Change in predicted class ratio | > 10 % deviation |
| **Business KPI** | Revenue / churn / default rate | Defined by business |

In [ ]:
# Simulate model performance monitoring over time
np.random.seed(3)
months        = pd.date_range("2024-01", periods=12, freq="MS")
baseline_acc  = 0.87
acc_over_time = np.cumsum(np.random.normal(0, 0.01, 12)).clip(-0.15, 0)
monthly_acc   = (baseline_acc + acc_over_time).clip(0.5, 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(months, monthly_acc, "b-o", label="Monthly Accuracy")
ax.axhline(y=baseline_acc,         color="green", linestyle="--", label="Baseline")
ax.axhline(y=baseline_acc - 0.05,  color="red",   linestyle="--", label="Alert threshold")
ax.fill_between(months, monthly_acc, baseline_acc - 0.05,
                where=(monthly_acc < baseline_acc - 0.05),
                alpha=0.3, color="red", label="Below threshold")
ax.set_title("Model Performance Monitoring — Monthly Accuracy")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

---

### Student Task 4.4

Build a complete end-to-end ML solution for the **customer churn** dataset:

1. Create a `Pipeline` that includes `StandardScaler` and `RandomForestClassifier`.
2. Use `GridSearchCV` to tune `n_estimators` (50, 100) and `max_depth` (3, 5, 10).
3. Save the best model using `joblib`.
4. Load the saved model and make a prediction for a **new, unseen customer** you define.
5. Summarise the model in one paragraph as if you were presenting to a non-technical business manager.

In [ ]:
# Your code here

---

### Evaluation Questions 4.4

1. What is the primary benefit of using an sklearn `Pipeline`?
   a) It automatically improves model accuracy
   b) It chains preprocessing and modelling into one reproducible object (correct)
   c) It replaces the need for cross-validation
   d) It speeds up data loading

2. In `GridSearchCV`, the parameter `cv=5` means:
   a) Only 5 hyperparameter combinations are tested
   b) The model is evaluated using 5-fold cross-validation (correct)
   c) Training runs for 5 epochs
   d) The best 5 features are selected

3. **Data drift** occurs when:
   a) The model's code has a bug
   b) The distribution of input features changes over time (correct)
   c) The model is retrained too frequently
   d) The training data has too many rows

4. Which `joblib` function saves a trained model to disk?
   a) `joblib.save()`
   b) `joblib.export()`
   c) `joblib.store()`
   d) `joblib.dump()` (correct)

5. Why should a deployed ML model be **retrained periodically**?
   a) To increase the size of the training dataset automatically
   b) Because older models always have bugs that need fixing
   c) Because real-world data distributions change over time, causing model performance to degrade (correct)
   d) sklearn models expire after 12 months

---